# 基礎編１
このノートブックでは、ブロックチェーンにアクセスする基本形の例を示します。
1. ダッシュボードの参照
2. スマートコントラクトの呼び出し
3. トランザクションログの参照

## 準備
実行の準備を行います。

DNCWARE/Blockchain+のクライアントライブラリを読み込みます。

In [1]:
var { api } = await import('../lib/load-blockchain-api.mjs');

設定ファイル`samples/etc/*.json`から、設定値を読み込みます。

In [2]:
var { password, peerURL, chainID, domain } = await import('../lib/load-config.mjs');
var { adminWalletJSON } = await import('../lib/load-wallet.mjs');

パスワードpasswordを使ってウォレットを開きます。  

In [3]:
var adminWallet = await api.unlockWalletFile(await api.parseWalletFile(adminWalletJSON), password);
console.log('wallet address:', adminWallet.address);

wallet address: eSQgkupyuMPkx22SuEui6HjUHyvTqm


ブロックチェーンに接続し、スマートコントラクトを呼び出す準備をします。

In [4]:
var rpc = new api.RPC(chainID);
rpc.connect(peerURL);

SocketHTTP {
  _triggers: Set(0) {},
  url: 'https://trial1.dncware-blockchain.biz',
  options: { agent: undefined, ca: undefined }
}

ノートブックで共通的に使用するユーティリティを読み込みます。

In [5]:
var { deploySmartContract } = await import('../lib/notebook-util.mjs');

## ダッシュボードの参照
ブロックチェーンに接続し、ダッシュボードの情報を読み出します。

In [6]:
var resp = await rpc.call(adminWallet, 'c1query', { type: 'dashboard' });
console.log('c1query dashboard:', JSON.stringify(resp.value));

c1query dashboard: {"N":3,"F":1,"B":0,"num_transactions":701641,"num_blocks":73222,"num_users":46947,"num_contracts":881,"num_groups":147,"num_domains":169}


## スマートコントラクトの呼び出し
スマートコントラクトをブロックチェーンにデプロイし、それを実行します。

ユーティリティ関数deploySmartContractを使って、スマートコントラクトをデプロイします。  
デプロイするスマートコントラクトbasic1は、引数aに対して処理結果としてa+1を返します。

In [7]:
var cid = await deploySmartContract({ a: 'number'}, function basic1(a){
   return a + 1;
});
console.log('contract id:', cid);

contract id: c002438689


引数aに数値12345を渡して、上でデプロイしたスマートコントラクトを実行します。  
応答のvalueには、12346が返ります。  
応答のtxnoにはトランザクション番号が、txidにはトランザクションIDが返ります。  

In [8]:
var resp1 = await rpc.call(adminWallet, cid, { a: 12345 });
console.log(resp1);

{
  txno: 701645,
  txid: 'xc4Sk9vZdmYMtqhW9moUjudssaTWDViv8novJcmJpENdHB',
  status: 'ok',
  value: 12346
}


引数aに数値999を渡して、上でデプロイしたスマートコントラクトを再度実行します。  
応答のvalueには、1000が返ります。

In [9]:
var resp2 = await rpc.call(adminWallet, cid, { a: 999 });
console.log(resp2);

{
  txno: 701646,
  txid: 'x3gYXcRLDWtCHyyUZ2Gyz5MWZcqFjcm4R2wB22sVEWV3o',
  status: 'ok',
  value: 1000
}


引数aに文字列を渡した場合には、引数の型に合わないので、応答のstatusにdeniedが返ります。

In [10]:
var resp3 = await rpc.call(adminWallet, cid, { a: 'xyz' });
console.log(resp3);

{
  txno: 701647,
  txid: 'xYdqcS9zxSVqenCcpSYbBUechNwg8RZwaK9xzFw2fXpbFB',
  status: 'denied',
  value: 'arg type: a'
}


## トランザクションの参照
上で実行した３つのトランザクションを参照します。

１つ目のトランザクションをtxnoを使って参照します。

In [11]:
var resp = await rpc.call(adminWallet, 'c1query', { type: 'a_transaction', id: resp1.txno });
console.log(resp);

{
  txid: 'xXsRHHEMtVFehhM4GEe9RMBJxrj26hnuWjL7wBJLrLXYEB',
  status: 'ok',
  value: {
    blkno: 73226,
    time: 1753420459462,
    txid: 'xc4Sk9vZdmYMtqhW9moUjudssaTWDViv8novJcmJpENdHB',
    addr: 'eSQgkupyuMPkx22SuEui6HjUHyvTqm',
    caller: [ 'u58281888', 'jupyter@c.TDSL.H011' ],
    callee: [ 'c002438689', 'basic1@c.TDSL.H011' ],
    status: 'ok',
    pack64: 'BQAAAAEAAADXAAAAAgAAAEAAAABAAAAAA3siY29udHJhY3QiOiJjMDAyNDM4Njg5IiwiYXJncyI6eyJhIjoxMjM0NX0sImFkZHIiOiJlU1Fna3VweXVNUGt4MjJTdUV1aTZIalVIeXZUcW0iLCJkZWFkbGluZSI6MTc1MzQyMDU1OTM1MSwib3JhY2xlIjoiTEJkOFlQQVZMeFBpQThlWiIsImJsb2NrcmVmIjp7Im5vIjo3MzIyMiwiaGFzaCI6IlRuVm43U3N1RC9iM0dPOVF6dEFkRUVlNWkxd1UzK3RLbnJ4M3cyUjEzMnM9In19ZXPxw2c37mebNx0UQDymdZOqotpzFW/twtDQIfAzQwYU8S32TPHQS3rATi17NeqF50z7jOEkewC5O78szMWyGk54j/HIapt/vhMgNLFfQhWCwGy6z5pFK3nQuo+p8KwDgdlJuYwfgfPZ6bYM0otS2fHZSYPGoQucPZUaVQwkoj0vzg==',
    txno: 701645,
    caller_txno: 0,
    argstr: '{"a":12345}',
    valuestr: '12346',
    subtxs: 0,
    steps: 106,
    disclosed

２つ目のトランザクションをtxidを使って参照します。

In [12]:
var resp = await rpc.call(adminWallet, 'c1query', { type: 'a_transaction', id: resp2.txid });
console.log(resp);

{
  txid: 'x7TH97k2JzXfYaDyGuezew53oFhiUAD2Vo6bkUtpnKgRw',
  status: 'ok',
  value: {
    blkno: 73227,
    time: 1753420460037,
    txid: 'x3gYXcRLDWtCHyyUZ2Gyz5MWZcqFjcm4R2wB22sVEWV3o',
    addr: 'eSQgkupyuMPkx22SuEui6HjUHyvTqm',
    caller: [ 'u58281888', 'jupyter@c.TDSL.H011' ],
    callee: [ 'c002438689', 'basic1@c.TDSL.H011' ],
    status: 'ok',
    pack64: 'BQAAAAEAAADVAAAAAgAAAEAAAABAAAAAA3siY29udHJhY3QiOiJjMDAyNDM4Njg5IiwiYXJncyI6eyJhIjo5OTl9LCJhZGRyIjoiZVNRZ2t1cHl1TVBreDIyU3VFdWk2SGpVSHl2VHFtIiwiZGVhZGxpbmUiOjE3NTM0MjA1NTk5MzAsIm9yYWNsZSI6IloyUkdROG9NdlJhdXE2UEgiLCJibG9ja3JlZiI6eyJubyI6NzMyMjYsImhhc2giOiJaM1pPaDBnRHp4UUFtWUR1RWE2a3Q2d3h3OS8rV2xrdytZVGZZNE01ZjlrPSJ9fWVz8cNnN+5nmzcdFEA8pnWTqqLacxVv7cLQ0CHwM0MGFPEt9kzx0Et6wE4tezXqhedM+4zhJHsAuTu/LMzFshpOeMAJMtkkPQcZCLth7dy99jlhMEYeCfHYR4F1HkbZRTm5JrJq1xOuFlPVuMVuS+CK8DZczDhDJvtjNDKRuSgO9ik=',
    txno: 701646,
    caller_txno: 0,
    argstr: '{"a":999}',
    valuestr: '1000',
    subtxs: 0,
    steps: 106,
    disclosed_to: [],


３つ目のエラーになったトランザクションも参照することができます。

In [13]:
var resp = await rpc.call(adminWallet, 'c1query', { type: 'a_transaction', id: resp3.txno });
console.log(resp);

{
  txid: 'xQiaxSWfiFstGCAvG9EEys2a4wkwXqf4DXQQpFig5hGwKB',
  status: 'ok',
  value: {
    blkno: 0,
    time: 1753420460600,
    txid: 'xYdqcS9zxSVqenCcpSYbBUechNwg8RZwaK9xzFw2fXpbFB',
    addr: 'eSQgkupyuMPkx22SuEui6HjUHyvTqm',
    caller: [ 'u58281888', 'jupyter@c.TDSL.H011' ],
    callee: [ 'c002438689', 'basic1@c.TDSL.H011' ],
    status: 'denied',
    pack64: 'BQAAAAEAAADXAAAAAgAAAEAAAABAAAAAA3siY29udHJhY3QiOiJjMDAyNDM4Njg5IiwiYXJncyI6eyJhIjoieHl6In0sImFkZHIiOiJlU1Fna3VweXVNUGt4MjJTdUV1aTZIalVIeXZUcW0iLCJkZWFkbGluZSI6MTc1MzQyMDU2MDQ5Mywib3JhY2xlIjoiRkZvZ2hjVGFDVmg5UUpCQiIsImJsb2NrcmVmIjp7Im5vIjo3MzIyNywiaGFzaCI6Iks2UWl0emFmYjZnUWRNRXBPMDdBam9ORkU5L1FEWmp1QkNuYWhsc3MrWGM9In19ZXPxw2c37mebNx0UQDymdZOqotpzFW/twtDQIfAzQwYU8S32TPHQS3rATi17NeqF50z7jOEkewC5O78szMWyGk54SkbgTHiAthVlNUkQFlHRP3JFiUv9ZZ3DU1ESq9J/8O+5WSC96PrQgrd/FuR5jGGK18OB+Ji6OvLXtCJm2gk5kA==',
    txno: 701647,
    caller_txno: 0,
    argstr: '{"a":"xyz"}',
    valuestr: '"arg type: a"',
    subtxs: 0,
    steps: 0,
    dis